# 3.3 线性回归简洁实现

In [1]:
import torch
from torch import nn
from torch.utils import data # 数据加载模块
from d2l import torch as d2l 

## 1. 生成数据

沿用上一节的数据生成方式，只是改用框架封装好的函数。
- d2l.synthetic_data 会返回 features 和 labels
- true_w 和 true_b 为数据集真实参数

In [2]:
true_w = torch.tensor([2, -3.4])
true_b = 4.2
features, labels = d2l.synthetic_data(true_w, true_b, 1000)

## 2. 读取数据集

使用 DataLoader 生成小批量数据。
- TensorDataset 把 features 和 labels 绑定
- DataLoader 负责按 batch 取样并可选打乱

In [3]:
def load_array(data_arrays, batch_size, is_train=True):
    dataset = data.TensorDataset(*data_arrays)
    return data.DataLoader(dataset, batch_size, shuffle=is_train)

batch_size = 10
data_iter = load_array((features, labels), batch_size)

快速查看一个小批量，确认迭代器正常工作。

In [4]:
next(iter(data_iter))

[tensor([[-2.5606, -1.4970],
         [ 0.0867, -0.3231],
         [-0.8130,  0.2332],
         [-0.8497,  1.0941],
         [-0.6134, -0.1688],
         [-1.3341, -1.1115],
         [-2.7071, -0.7984],
         [ 2.1408,  0.2784],
         [ 0.1790, -0.1379],
         [-0.9407, -0.1240]]),
 tensor([[ 4.1782],
         [ 5.4650],
         [ 1.7919],
         [-1.2263],
         [ 3.5566],
         [ 5.3139],
         [ 1.5130],
         [ 7.5370],
         [ 5.0420],
         [ 2.7500]])]

## 3. 定义模型

用 nn.Linear 构建线性回归层，外面包一层 Sequential，以便后续添加更多层。
- 输入特征数为 2
- 输出为 1 个标量

In [5]:
net = nn.Sequential(nn.Linear(2, 1))

## 4. 初始化参数

- 权重从正态分布采样
- 偏置初始化为 0

In [6]:
net[0].weight.data.normal_(0, 0.01)
net[0].bias.data.fill_(0)

tensor([0.])

## 5. 定义损失和优化器

- 使用均方误差 MSELoss
- 使用 SGD 优化器，学习率设为 0.03

In [7]:
loss = nn.MSELoss()
trainer = torch.optim.SGD(net.parameters(), lr=0.03)

## 6. 训练

训练过程与从零实现类似：
- net(X) 直接得到预测值
- loss 自动求平均
- trainer.step() 自动更新参数

In [8]:
num_epochs = 3
for epoch in range(num_epochs):
    for X, y in data_iter:
        l = loss(net(X), y)
        trainer.zero_grad()
        l.backward()
        trainer.step()
    l = loss(net(features), labels)
    print(f'epoch {epoch + 1}, loss {l:f}')

epoch 1, loss 0.000272
epoch 2, loss 0.000097
epoch 3, loss 0.000096


## 7. 对比学到的参数

训练完成后，学到的参数应接近真实值。

In [9]:
w = net[0].weight.data
b = net[0].bias.data
print('w的估计误差：', true_w - w.reshape(true_w.shape))
print('b的估计误差：', true_b - b)

w的估计误差： tensor([-9.7513e-05, -2.9635e-04])
b的估计误差： tensor([8.6784e-05])


## 小结

- 使用 nn 和 optim 能显著减少代码量
- DataLoader 统一了小批量读取逻辑
- 高级 API 让训练流程更清晰、更稳定